In [0]:
#imports
from pyspark.sql.functions import year,sum


In [0]:
# read the processed data

df_customers_processed = spark.read.format("delta").table("processed.customers")
df_orders_processed = spark.read.format("delta").table("processed.orders")
df_products_processed = spark.read.format("delta").table("processed.products")

In [0]:
#pick only required columns

df_orders_selected = df_orders_processed.select("order_date", "profit", "customer_id", "product_id")
df_customers_selected = df_customers_processed.select("customer_id", "customer_name", "country")
df_products_selected = df_products_processed.select("product_id", "category", "sub_category")

df_orders_customers_joined = df_orders_selected.join(df_customers_selected, "customer_id", "inner")
df_orders_customers_products_joined = df_orders_customers_joined.join(df_products_selected, "product_id", "inner")

#display(df_orders_customers_products_joined)

In [0]:
#load data into processed layer

df_orders_customers_products_joined.write.mode("overwrite").format("delta").option("overwriteschema",True).saveAsTable("processed.ecomm_combined_data")

In [0]:
#aggregating profit on year, category, sub_category and customer

df_profit_agg = df_orders_customers_products_joined.groupBy(
    year("order_date").alias("order_year"),
    "category",
    "sub_category",
    "customer_id"
).agg(
    sum("profit").alias("total_profit")
).orderBy(
    "order_year",
    "category",
    "sub_category",
    "customer_id"
)

df_profit_agg.write.mode("overwrite").format("delta").saveAsTable("presentation.ecomm_profit_agg")


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
display(df_profit_agg)

Databricks data profile. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.